# 🚀 Otimização de Hiperparâmetros - CatBoost
Este notebook detalha o processo de modelagem e ajuste fino para o **Diabetes Prediction Challenge**, utilizando o algoritmo **CatBoost** integrado a uma estratégia de busca estocástica para maximizar a capacidade preditiva do modelo.

## 🛠️ Estratégia de Modelagem
* **Pipeline de Dados:** Utilização do pré-processador customizado `v3a` (carregado via `joblib`), garantindo que transformações como *Target Encoding* e normalização sejam aplicadas consistentemente.
* **Métrica de Sucesso:** O foco principal da otimização é o ROC AUC, dado que o desafio exige a predição da probabilidade de diagnóstico (`diagnosed_diabetes`) e o dataset apresenta um leve desbalanceamento.
* **Eficiência de Treinamento:** Implementação de *Early Stopping* e uso de um conjunto de validação dedicado para prevenir o *overfitting* durante as iterações de busca.

## 🔍 Fluxo de Otimização (O Funil)
1.  **Baseline (Referência):** Treinamento do CatBoost com parâmetros padrão (*default*) para estabelecer um marco de performance (`R0`) e validar o pipeline.
2.  **Randomized Search (Exploração):** Execução de `RandomizedSearchCV` com 36 iterações, mapeando um espaço multidimensional que inclui:
    * **Boosting:** Número de iterações e taxa de aprendizado (`learning_rate`).
    * **Estrutura:** Profundidade das árvores (`depth`) e regularização (`l2_leaf_reg`).
    * **Diversidade:** Parâmetros de *bagging* e força de aleatoriedade para robustez do modelo.
3.  **Avaliação Comparativa:** Confronto direto entre o modelo Baseline e o modelo Otimizado (`R1`) utilizando métricas de erro, matriz de confusão e curvas ROC.

## 📊 Critérios de Sucesso
* **Generalização:** Validação cruzada com `StratifiedKFold` (3 splits) para garantir estabilidade estatística em um dataset de 700.000 linhas.
* **Persistência:** Exportação do melhor estimador encontrado para uso em produção ou submissão final no Kaggle.


In [1]:
import warnings
warnings.simplefilter("ignore", FutureWarning)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import time

# sklearn

from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.experimental import enable_halving_search_cv 

from sklearn.model_selection import (
    train_test_split, KFold, cross_validate, cross_val_score,
    RandomizedSearchCV, StratifiedKFold,train_test_split,HalvingRandomSearchCV
)

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             average_precision_score,roc_curve)


from sklearn.base import BaseEstimator, TransformerMixin, clone
from scipy.stats import randint, uniform, loguniform

# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.preprocess_utils_diab3a import * #(NOVO atualizações)

print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))
start_inicial = time.time()


#Processo iniciado em: 22:43:32


## 1. Load Data & Pipeline

In [2]:
BASE = Path.cwd().parent

PP3a = joblib.load(BASE/'src'/'preprocess_diabetes_v3a.joblib')['preprocessador']

DATA_DIR = BASE/"data"/"raw"
X_train = pd.read_csv(DATA_DIR/"X_train_raw.csv")
X_val  = pd.read_csv(DATA_DIR/"X_test_raw.csv")
y_train = pd.read_csv(DATA_DIR/"y_train_raw.csv").values.ravel()
y_val  = pd.read_csv(DATA_DIR/"y_test_raw.csv").values.ravel()

mtd_scoring='roc_auc'
print("\n#Processo em:", time.strftime("%H:%M:%S"))


#Processo em: 22:43:33


## 2.Baseline

In [3]:
# MODEL BASE
print("#Processo iniciado em:", time.strftime("%H:%M:%S"))
categorical_features = ['gender','ethnicity', 'smoking_status','employment_status', 'Age_Group','PAW_Group']
model_base = CatBoostClassifier(loss_function='Logloss',eval_metric='AUC',random_state=42,
                               verbose=False,early_stopping_rounds=100,cat_features=categorical_features,nan_mode='Min') 
cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

pipe_base  = pipe_models(model_base, PP3a)

R0=evaluate_model(pipe_base,X_train,y_train,X_val,y_val,modelname='Baseline')   

# =====================================================
# 6) Hiperparametros utilizados
# =====================================================
print(f"{'='*70}")
print_hiper(model_base.get_params())
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))


#Processo iniciado em: 22:43:33

                         📍 RESULTADOS BASELINE                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7246 ± 0.0011

✅ TEST SET
   Padrão (0.5):                  0.6832
   Otimizado:                     0.6831 (threshold = 0.510)
   ROC-AUC:                       0.7252
   Avg precision:                 0.8102
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Loss Function             : Logloss
• Nan Mode                  : Min
• Verbose                   : False
• Eval Metric               : AUC
• Random State              : 42
• Early Stopping Rounds     : 100
• Cat Features              : ['gender', 'ethnicity', 'smoking_status', 'employment_status', 'Age_Group', 'PAW_Group']
--------------------------------------------------

#Processo finalizado em: 22:58:07


In [4]:
import time
from scipy.stats import randint, loguniform, uniform

# EARLY STOPPING - CATBOOST VERSION
print("#Processo Iniciado em:", time.strftime("%H:%M:%S"))

# =====================================================
# a) Split de validação
# =====================================================
X_search, X_vals, y_search, y_vals = train_test_split(X_train, y_train, test_size=0.10, stratify=y_train, random_state=42)
# O CatBoost dentro do Pipeline precisa que o X_val seja processado pelo PP3a ANTES
X_val_p = PP3a.transform(X_vals)

# =====================================================
# b) Definição do hiperparametro de Busca (CatBoost Specific)
# =====================================================
param_dist_1 = {
    # Boosting
    'model__iterations': randint(500, 2000),
    'model__learning_rate': loguniform(0.05, 0.25),
    
    # Estrutura
    'model__depth': [4,5,6,7,8],
    'model__l2_leaf_reg': [1,2,3,4,5],
    
    # Diversidade / Aleatoriedade
    'model__bagging_temperature': uniform(0, 1),
    'model__random_strength': uniform(0.5, 5),
    'model__border_count': [128, 254],
}

# =====================================================
# c) Definição do Buscador 
# =====================================================
numero_itera = 36 
search_c = RandomizedSearchCV(
    pipe_base,
    param_dist_1,
    n_iter=numero_itera,
    cv=cv_s,
    scoring=mtd_scoring,
    random_state=42,
    verbose=0) # Verbose 1 para acompanhar o progresso das iterações

# =====================================================
# d) Buscando melhor conjunto de parametros 
# =====================================================
# Nota: No CatBoost via sklearn, os parâmetros de fit do modelo 
# devem ser passados com o prefixo 'model__'
start = time.time()
search_c.fit(
    X_search, y_search,
    model__eval_set=(X_val_p, y_vals), 
    model__use_best_model=True,
    model__verbose=False
)
end = time.time()

# -----------------------------------------------------
# 1) Predição e Avaliação Final
# -----------------------------------------------------
best_c = search_c.best_estimator_

R1=evaluate_model(best_c,X_train,y_train,X_val,y_val,modelname='search',pipe_fit=False)   

print(f"{'='*70}")
print("🎯 Melhores Parâmetros Encontrados:")
print_hiper(search_c.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

#Processo Iniciado em: 22:58:07

                          📍 RESULTADOS SEARCH                         
📊 CROSS-VALIDATION
   Média roc_auc:                0.7248 ± 0.0013

✅ TEST SET
   Padrão (0.5):                  0.6835
   Otimizado:                     0.6834 (threshold = 0.520)
   ROC-AUC:                       0.7254
   Avg precision:                 0.8105
🎯 Melhores Parâmetros Encontrados:
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Bagging Temperature       : 0.8287375091519293
• Border Count              : 254
• Depth                     : 4
• Iterations                : 1263
• L2 Leaf Reg               : 4
• Learning Rate             : 0.11975627749563679
• Random Strength           : 1.2046211248738132
--------------------------------------------------

#Processo finalizado em: 03:59:19


In [5]:
# # salvar modelos
#joblib.dump(pipe_base, 'modelo_CBT_final_base.'+mtd_scoring+'_v3a.joblib')
joblib.dump(search_c.best_estimator_, 'modelo_CBT_final_randsearch.'+mtd_scoring+'_v3a.joblib')
print(f"✅ Arquivo salvo • {time.strftime('%d/%m/%Y-%H:%M:%S')}")

✅ Arquivo salvo • 11/03/2026-03:59:19
